In [1]:
model_name = "urchade/gliner_multi-v2.1"

In [2]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
import json 
import random 


In [3]:
train_path = "100k_aya_expanse_gliner.json"
# train_path = "../data.json"

with open(train_path,"r") as f:
    data = json.load(f)

print('Dataset size:', len(data))

random.shuffle(data)
print('Dataset is shuffled...')

train_dataset = data[:int(len(data)*0.9)]
test_dataset = data[int(len(data)*0.9):]

print('Dataset is splitted...')


Dataset size: 86116
Dataset is shuffled...
Dataset is splitted...


In [4]:
train_dataset[0].keys()

dict_keys(['tokenized_text', 'ner'])

In [5]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import torch
torch.cuda.set_device('cuda:0')
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset

In [7]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

model = GLiNER.from_pretrained(model_name)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [8]:
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)


In [9]:
model.to(device)
print("done")

done


In [8]:

import torch
torch.cuda.empty_cache()
print("Available GPUs:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
# print("Trainer args:", training_args.device)

Available GPUs: 1
Current device: 0


In [11]:
num_steps = 500
batch_size =8
data_size = len(train_dataset)
num_batches = data_size // batch_size
num_epochs = max(1, num_steps // num_batches)

training_args = TrainingArguments(
    output_dir="models",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear", #cosine
    warmup_ratio=0.1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    num_train_epochs=num_epochs,
    evaluation_strategy="steps",
    save_steps = 100,
    save_total_limit=10,
    dataloader_num_workers = 0,
    use_cpu = False,
    report_to="none",
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

trainer.train()

/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/transformers/training_args.py:1609: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_924711/3557259569.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Step,Training Loss,Validation Loss
500,18.182900,246.791870
1000,11.006600,246.292709
1500,9.951600,196.399719
2000,9.190000,222.525406


/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/gliner/data_processing/processor.py:296: UserWarning: Sentence of length 404 has been truncated to 384
  warnings.warn(f"Sentence of length {len(tokens)} has been truncated to {max_len}")
/home/ai/miniconda3/envs/nlp/lib/python3.11/site-packages/gliner/data_processing/processor.py:296: UserWarning: Sentence of length 503 has been truncated to 384
  warnings.warn(f"Sentence of length {len(tokens)} has been truncated to {max_len}")


TrainOutput(global_step=2209, training_loss=11.903900326080805, metrics={'train_runtime': 1135.4082, 'train_samples_per_second': 15.564, 'train_steps_per_second': 1.946, 'total_flos': 0.0, 'train_loss': 11.903900326080805, 'epoch': 1.0})

In [9]:
trained_model = GLiNER.from_pretrained("models/checkpoint-2209", load_tokenizer=True)


config.json not found in /home/ai/kobo/bert_world/gliner_world/models/checkpoint-2209


In [10]:
texts = [
    """
    فاز ليونيل ميسي بجائزة الكرة الذهبية لعام 2023 بعد أداء مذهل مع نادي باريس سان جيرمان ومنتخب الأرجنتين في كأس العالم. يُعتبر ميسي من أفضل لاعبي كرة القدم في العالم، وقد قاد فريقه للفوز بلقب الدوري الفرنسي.
    """,
    """
    في عام 1945، انتهت الحرب العالمية الثانية بعد استسلام ألمانيا. وقّع الحلفاء اتفاقية في باريس، وأصبحت الأمم المتحدة رمزًا للسلام العالمي.
    """,
    
    """
    أعلنت شركة جوجل عن إطلاق منتج جديد في مؤتمرها السنوي في كاليفورنيا. المنتج الجديد، الذي طوره فريق بقيادة سوندار بيتشاي، يهدف إلى تحسين تجربة المستخدم.
    """,
    
    """
    نال الكاتب نجيب محفوظ جائزة نوبل للآداب عام 1988 عن روايته "أولاد حارتنا". تُرجم العمل إلى عدة لغات، وأُقيم احتفال كبير في القاهرة لتكريمه.
    """,
    
    """
    فازت السعودية باستضافة معرض إكسبو 2030 بعد منافسة قوية مع كوريا الجنوبية. سيُقام الحدث في الرياض، وسيشارك فيه عدد كبير من الشركات العالمية مثل أمازون ومايكروسوفت.
    """
]
labels = ["Person", "Award", "Organization", "Location", "Event"]

for i, text in enumerate(texts, 1):
    print(f"\nاختبار النص {i}:")
    entities = trained_model.predict_entities(text, labels, threshold=0.5)
    for entity in entities:
        print(entity["text"], "=>", entity["label"])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



اختبار النص 1:
ليونيل ميسي => Person
الكرة الذهبية لعام 2023 => Award
نادي باريس سان جيرمان => Organization
كأس العالم => Event
العالم => Location
الدوري الفرنسي => Location

اختبار النص 2:
الحرب العالمية الثانية => Event
الحلفاء => Organization
باريس => Location
الأمم المتحدة => Organization
رمزًا للسلام العالمي => Award

اختبار النص 3:
جوجل => Organization
مؤتمرها السنوي => Event
كاليفورنيا => Location
سوندار بيتشاي => Person

اختبار النص 4:
نجيب محفوظ => Person
احتفال كبير => Event
القاهرة => Location

اختبار النص 5:
السعودية => Location
إكسبو 2030 => Event
كوريا الجنوبية => Organization
الرياض => Location
أمازون => Organization
ومايكروسوفت => Organization


In [13]:
trained_model.push_to_hub("gliner_arabic-v2.1")

pytorch_model.bin:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Abdelkareem/gliner_arabic-v2.1/commit/fc13cc567dd4d4dd508fc126df39980c5313d625', commit_message='Push model using huggingface_hub.', commit_description='', oid='fc13cc567dd4d4dd508fc126df39980c5313d625', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Abdelkareem/gliner_arabic-v2.1', endpoint='https://huggingface.co', repo_type='model', repo_id='Abdelkareem/gliner_arabic-v2.1'), pr_revision=None, pr_num=None)

In [6]:
print(train_dataset[0])  # Show the first record

{'tokenized_text': ['ت', 'ُ', 'عتبر', 'شواطئ', 'البحر', 'الأحمر', 'وجهة', 'سياحية', 'شهيرة', 'للسياح', 'الباحثين', 'عن', 'سياحة', 'المغامرات', '.', 'حيث', 'يستمتع', 'الزوار', 'برحلات', 'بحرية', 'باستخدام', 'سياراتهم', 'الخاصة', 'خلال', 'فصل', 'الصيف', '.'], 'ner': [[3, 5, 'اسم الوجهة السياحية'], [25, 25, 'الموسم السياحي']]}


In [7]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Convert train_dataset to a DataFrame
train_df = pd.DataFrame(train_dataset)

# Convert test_dataset to a DataFrame (assuming it has the same structure)
test_df = pd.DataFrame(test_dataset)

# Inspect the DataFrame
print(train_df.head())

                                      tokenized_text  \
0  [ت, ُ, عتبر, شواطئ, البحر, الأحمر, وجهة, سياحي...   
1  [تعتبر, شانيل, من, أشهر, الماركات, العالمية, ف...   
2  [يعتبر, الحكيم, خالد, بن, يوسف, من, أشهر, الأط...   
3  [في, مسابقة, نهائية, مثيرة, ،, فاز, فريق, الات...   
4  [ألقى, الإمام, عبد, الرحمن, الدوسري, خطبة, الج...   

                                                 ner  
0  [[3, 5, اسم الوجهة السياحية], [25, 25, الموسم ...  
1  [[10, 11, اسم المنتج], [1, 1, الماركة], [14, 1...  
2  [[1, 4, اسم الطبيب], [19, 20, العلاج التقليدي]...  
3      [[6, 7, اسم الفريق], [14, 15, اللاعب المميز]]  
4  [[1, 4, اسم العالم], [11, 11, الموضوع الديني],...  


In [8]:
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value
from huggingface_hub import login

import pandas as pd
from datasets import Dataset, DatasetDict, Features, Sequence, Value



In [49]:
print(train_df.iloc[0])

tokenized_text    [ي, ُ, قام, مهرجان, الجنادرية, في, الإمارات, خ...
ner               [{'start': 3, 'end': 4, 'label': 'اسم الفعالية...
Name: 0, dtype: object


In [9]:
import pandas as pd
from datasets import Dataset


def convert_ner_to_dict(ner_list):
    return [
        {"start": entity[0], "end": entity[1], "label": entity[2]}
        for entity in ner_list
    ]


train_df["ner"] = train_df["ner"].apply(convert_ner_to_dict)


In [11]:

test_df["ner"] = test_df["ner"].apply(convert_ner_to_dict)

# Convert the DataFrame to a Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Push to Hugging Face Hub
# train_dataset.push_to_hub("100k_aya_expanse_ner_gliner")
hf = DatasetDict({"train": train_dataset, "test_dataset": test_dataset})

In [12]:
hf.push_to_hub("100k_aya_expanse_ner_gliner")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/78 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Abdelkareem/100k_aya_expanse_ner_gliner/commit/8fcb07012fa61b1e6267d03d6b20926b14982d5d', commit_message='Upload dataset', commit_description='', oid='8fcb07012fa61b1e6267d03d6b20926b14982d5d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Abdelkareem/100k_aya_expanse_ner_gliner', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Abdelkareem/100k_aya_expanse_ner_gliner'), pr_revision=None, pr_num=None)